Data Acquisition & Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/balanced_log_dataset.csv")

def parse_event_sequence(x):
    if pd.isna(x):
        return []
    x = x.strip().strip("[]")
    return [e.strip() for e in x.split(",") if e.strip()]

df['Features'] = df['Features'].apply(parse_event_sequence)

df['Label'] = df['Label'].map({'Success': 0, 'Fail': 1})
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Latency_norm'] = scaler.fit_transform(df[['Latency']])


print(df['Features'].head())
print(df['Label'].value_counts())


0    [E5, E5, E5, E22, E11, E9, E11, E9, E11, E9, E...
1    [E22, E5, E5, E5, E26, E26, E11, E9, E11, E9, ...
2    [E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...
3    [E5, E5, E22, E5, E11, E9, E11, E9, E26, E26, ...
4    [E5, E5, E22, E5, E9, E11, E9, E11, E9, E26, E...
Name: Features, dtype: object
Label
1    16838
0    16838
Name: count, dtype: int64


In [ ]:
all_events = set(e for seq in df['Features'] for e in seq)

event2idx = {event: idx + 1 for idx, event in enumerate(all_events)}
event2idx['PAD'] = 0

vocab_size = len(event2idx)


In [ ]:
MAX_LEN = 50

def encode_sequence(seq):
    seq = [event2idx[e] for e in seq]
    return seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(seq))

df['encoded'] = df['Features'].apply(encode_sequence)


Feature Engineering & Model Definition

In [ ]:
X = torch.tensor(df['encoded'].tolist(), dtype=torch.long)
latency = torch.tensor(df['Latency_norm'].values, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)

X_train, X_test, lat_train, lat_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    latency,
    y,
    df.index,            # ⭐ THIS IS THE KEY
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_test_idx = idx_test   # save for export



In [ ]:
class LogDataset(Dataset):
    def __init__(self, X, latency, y):
        self.X = X
        self.latency = latency
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.latency[idx], self.y[idx]



In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        # +1 for latency
        self.fc = nn.Linear(embed_dim + 1, 2)

    def forward(self, x, latency):
        # Padding mask (IMPORTANT)
        mask = (x == 0)

        x = self.embedding(x)
        x = self.encoder(x, src_key_padding_mask=mask)
        x = x.mean(dim=1)

        # concatenate latency
        x = torch.cat([x, latency.unsqueeze(1)], dim=1)

        return self.fc(x)



Training Pipeline & Optimization

In [ ]:
train_loader = DataLoader(
    LogDataset(X_train, lat_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    LogDataset(X_test, lat_test, y_test),
    batch_size=64
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerClassifier(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            return False  # continue training
        else:
            self.counter += 1
            return self.counter >= self.patience  # stop training




In [ ]:
EPOCHS = 20
PATIENCE = 3

early_stopper = EarlyStopping(patience=PATIENCE)

for epoch in range(EPOCHS):
    # ------------------
    # Training
    # ------------------
    model.train()
    train_loss = 0.0

    for xb, latb, yb in train_loader:
        xb = xb.to(device)
        latb = latb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model(xb, latb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ------------------
    # Validation
    # ------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for xb, latb, yb in test_loader:
            xb = xb.to(device)
            latb = latb.to(device)
            yb = yb.to(device)

            preds = model(xb, latb)
            loss = criterion(preds, yb)
            val_loss += loss.item()

    val_loss /= len(test_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # ------------------
    # Early stopping
    # ------------------
    if early_stopper.step(val_loss, model):
        print("⏹️ Early stopping triggered")
        break

# Restore best model
model.load_state_dict(early_stopper.best_state)
print("✅ Best model weights restored")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/20 | Train Loss: 0.0675 | Val Loss: 0.1354
Epoch 2/20 | Train Loss: 0.0083 | Val Loss: 0.1244
Epoch 3/20 | Train Loss: 0.0049 | Val Loss: 0.1117
Epoch 4/20 | Train Loss: 0.0049 | Val Loss: 0.1045
Epoch 5/20 | Train Loss: 0.0061 | Val Loss: 0.1642
Epoch 6/20 | Train Loss: 0.0099 | Val Loss: 0.1643
Epoch 7/20 | Train Loss: 0.0075 | Val Loss: 0.1265
⏹️ Early stopping triggered
✅ Best model weights restored
